In [5]:
import sys, torch
print(sys.executable)
print("Torch:", torch.__version__)
print("MPS:", torch.backends.mps.is_available())

/Users/apple/Desktop/BRAINIAC/brainiac_env/bin/python
Torch: 2.10.0
MPS: True


In [6]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [7]:
import torch
import numpy as np
...

Ellipsis

In [ ]:

import os, glob, random
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete


torch._dynamo.config.suppress_errors = True
torch._dynamo.reset()
torch.backends.cudnn.enabled = False


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


DATASET_PATH = "/Users/apple/Desktop/BRAINIAC/data/processed"
SAVE_DIR = "/Users/apple/Desktop/BRAINIAC/models"
os.makedirs(SAVE_DIR, exist_ok=True)

IMAGE_PATH = f"{DATASET_PATH}/images"
MASK_PATH  = f"{DATASET_PATH}/masks"

X_files = sorted(glob.glob(f"{IMAGE_PATH}/*.npy"))
y_files = sorted(glob.glob(f"{MASK_PATH}/*.npy"))

train_X, val_X, train_y, val_y = train_test_split(
    X_files, y_files, test_size=0.2, random_state=42
)

class BrainPatchDataset(Dataset):
    def __init__(self, X, Y, patch=80, tumor_ratio=0.7):
        self.X = X
        self.Y = Y
        self.patch = patch
        self.tumor_ratio = tumor_ratio

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = np.load(self.X[idx])
        y = np.load(self.Y[idx])

        if random.random() < self.tumor_ratio and y.sum() > 0:
            coords = np.argwhere(y > 0)
            z,y0,x0 = coords[np.random.randint(len(coords))]
        else:
            z = np.random.randint(0, y.shape[0])
            y0 = np.random.randint(0, y.shape[1])
            x0 = np.random.randint(0, y.shape[2])

        ps = self.patch
        z1 = np.clip(z-ps//2, 0, y.shape[0]-ps)
        y1 = np.clip(y0-ps//2, 0, y.shape[1]-ps)
        x1 = np.clip(x0-ps//2, 0, y.shape[2]-ps)

        x = x[z1:z1+ps, y1:y1+ps, x1:x1+ps]
        y = y[z1:z1+ps, y1:y1+ps, x1:x1+ps]

        x = torch.tensor(x, dtype=torch.float32).permute(3,0,1,2).contiguous()
        y = torch.tensor(y, dtype=torch.long).unsqueeze(0).contiguous()

        return x, y


train_loader = DataLoader(
    BrainPatchDataset(train_X, train_y, patch=96, tumor_ratio=0.7),
    batch_size=1,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    BrainPatchDataset(val_X, val_y, patch=96, tumor_ratio=0.5),
    batch_size=1,
    shuffle=False,
    num_workers=0
)


model = SwinUNETR(
    in_channels=3,
    out_channels=2,
    feature_size=12,
    use_checkpoint=False
).to(device)


loss_fn = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
    lambda_dice=1.5,
    lambda_ce=0.5,
    weight=torch.tensor([0.02,0.98]).to(device)
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

dice_metric = DiceMetric(include_background=False, reduction="mean")

post_pred  = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)


CHECKPOINT_PATH = f"{SAVE_DIR}/checkpoint.pt"
start_epoch = 0
best_dice = 0

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    best_dice = ckpt["best_dice"]
    print("Resumed from", start_epoch)


max_epochs = 80

for epoch in range(start_epoch, max_epochs):
    print(f"\nEpoch {epoch+1}/{max_epochs}")

    model.train()
    train_loss = 0

    for x,y in tqdm(train_loader):
        x,y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)

        y_hat = model(x)
        loss = loss_fn(y_hat,y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print("Train Loss:", train_loss/len(train_loader))

    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device), y.to(device)
            y_hat = model(x)

            y_hat = post_pred(y_hat)
            y = post_label(y)

            dice_metric(y_hat,y)

    val_dice = dice_metric.aggregate().item()
    print("Val Dice:", val_dice)

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(),
                   f"{SAVE_DIR}/best_swin_unetr.pt")
        print("Saved best model")

    torch.save({
        "epoch":epoch,
        "model_state":model.state_dict(),
        "optimizer_state":optimizer.state_dict(),
        "best_dice":best_dice
    }, CHECKPOINT_PATH)

print("Done. Best Dice:", best_dice)

Using device: mps
Resumed from 38

Epoch 39/80


100%|██████████| 294/294 [08:24<00:00,  1.72s/it]


Train Loss: 0.11934709732680499
Val Dice: 0.49172237515449524

Epoch 40/80


100%|██████████| 294/294 [08:21<00:00,  1.71s/it]


Train Loss: 0.11518391229364337
Val Dice: 0.49170148372650146

Epoch 41/80


...

Train Loss: 0.09017517092023171
Val Dice: 0.49171632528305054

Epoch 79/80


100%|██████████| 294/294 [08:24<00:00,  1.72s/it]


Train Loss: 0.08892948732895105
Val Dice: 0.49170905351638794

Epoch 80/80


100%|██████████| 294/294 [08:22<00:00,  1.71s/it]


Train Loss: 0.08768610820985165
Val Dice: 0.4916646182537079
Done. Best Dice: 0.49189284443855286
